In [0]:
df = spark.read.csv("/Volumes/databricks_practice/inputdb/customerdata/customers.csv",header=True,inferSchema=True)
df.display()
df.printSchema()

In [0]:
df =spark.read.options(header='True',inferSchema='True').csv("/Volumes/databricks_practice/inputdb/customerdata/customers.csv")
df.display()

In [0]:
df= spark.read.format("csv").options(header='True',inferSchema='True').load("/Volumes/databricks_practice/inputdb/customerdata/customers.csv")
df.display()
df1= df.filter(df.Age>30)
df1.display()

In [0]:
#custom schema to handle performance
import pyspark.sql.types as st
from pyspark.sql.types import *
from pyspark.sql.functions import *
cust_schema =StructType([
StructField("Name",StringType(),True),
StructField("City",StringType(),True),
StructField("Age",IntegerType(),True),
StructField("proof",StringType(),True)])
df = spark.read.schema(cust_schema).csv("/Volumes/databricks_practice/inputdb/customerdata/customers.csv")
df.display()
df1= df.filter(df.Age>30)
df1.display()

In [0]:
#muliple files reading
df=spark.read.format("csv").options(header='True',inferSchema='True').load("/Volumes/databricks_practice/inputdb/customerdata/*.csv")

In [0]:
#custom schema sql string to handle performance
import pyspark.sql.types as st
from pyspark.sql.types import *
from pyspark.sql.functions import *
cust_schema ="name string, city string, Age int,proof string"
df = spark.read.schema(cust_schema).csv("/Volumes/databricks_practice/inputdb/customerdata/customers.csv",header=True)
df.display()
df1= df.filter(df.Age>30)
df1.display()

### Options for handling double quotes

id,name,remarks<br>
1,"Ramesh, K.P","Good perfromer"<br>
2,"Manoj","Needs "special" attention"

In [0]:
df =spark.read.format("csv").options(header =True,inferschema =True,quote="\"",escape="\"").load("/Volumes/databricks_practice/inputdb/empdata/emp_perf.csv")
df.display()

###Multi line read
id,name,remarks<br>
1,"Ramesh, K.P","Good perfromer"<br>
2,"Manoj","Needs "special" attention<br>
and care for him"

In [0]:
df_muli = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .option("quote", "\"")
      .option("escape", "\"")
      .option("multiLine", "True")
      .load("/Volumes/databricks_practice/inputdb/empdata/emp_perf_multi_line.csv"))

df_muli.display()


## Read modes in csv

id,name,remarks<br>
1, IRamesh, K.P.","Good performer"<br>
2, "Manoj", "Needs "special" attention"<br>
3,"Test ""double quotes"" inside field"<br>
4,"Incomplete row<br>
5,"Extra","Column", "Here"<br>

##Permissive Mode (Default):
This is the default mode if no other mode is specified.
When encountering a malformed record, Spark sets the unparseable fields to null.
Additionally, it can place the entire corrupt record into a designated string column (by default, _corrupt_record) for later inspection or processing.
This mode prioritizes processing continuity, allowing the job to complete even with corrupt data.

## DropMalformed Mode:
In this mode, Spark discards any rows that contain malformed records.
Only records that fully conform to the expected schema are included in the resulting DataFrame or Dataset.
This mode is suitable when data integrity is paramount, and corrupt records are considered unusable.
## FailFast Mode:
This mode immediately throws an exception and fails the job as soon as it encounters a malformed record.
It ensures strict schema adherence and is used in scenarios where any data corruption is unacceptable and should halt the entire process.

In [0]:
emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("remarks", StringType(), True),
    StructField("_corrupt_record", StringType(), True)  
])
df_permissive = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .option("quote", "\"")
      .option("escape", "\"")
      .option("mode","PERMISSIVE")
      .option("schema",emp_schema)
      .load("/Volumes/databricks_practice/inputdb/empdata/emp_perf_corrupted.csv"))

df_permissive.display()


emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("remarks", StringType(), True)
])
df_malform = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .option("quote", "\"")
      .option("escape", "\"")
      .option("mode","DROPMALFORMED")
      .option("schema",emp_schema)
      .load("/Volumes/databricks_practice/inputdb/empdata/emp_perf_corrupted.csv"))

df_malform.display()


emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("remarks", StringType(), True)
])
df_fail_fast = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .option("quote", "\"")
      .option("escape", "\"")
      .option("mode","FAILFAST")
      .option("schema",emp_schema)
      .load("/Volumes/databricks_practice/inputdb/empdata/emp_perf_corrupted.csv"))

df_fail_fast.display()

